<a href="https://colab.research.google.com/github/internshipinabook/data-science-internshipinabook/blob/main/notebooks/week8_evaluation_STARTER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ⏸️ **Pause and Predict**
> Before setting your decision threshold: write down what false positives and false negatives cost in this business context. A customer incorrectly flagged for churn gets an unnecessary retention call. A missed churner leaves without intervention. Which is more expensive?

In [ ]:
# ── Colab setup (skip if running locally) ────────────────────────────────────
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/data-science-internshipinabook.git book
    %cd book
    !pip install -r requirements.txt -q
    print('Colab setup complete ✅')
else:
    print('Running locally ✅')


# Week 8 — Evaluation and Business Value
### Week 8 — The Data Science Internship | Kova Analytics

> **STARTER notebook.**

The Data Science Internship Book 0 of 9 in the InternshipInABook™ Series

This repository contains the practical exercises from the book.

📖 Complete internship experience:

Workplace scenarios
Mentor guidance
Reflection exercises
Portfolio
checkpoints
Capstone projects
👉 Get the complete book: https://selar.com/al990ay7ux

🚀 Start Here
Welcome to The Data Science Internship.

If you are new to the InternshipInABook™ Series:

Run this setup notebook.
Complete Week 1.
Follow the weekly internship schedule.
Build your portfolio.
Complete the capstone project.
🔗 Repository: https://github.com/internshipinabook/data-science-internshipinabook

Run every cell top to bottom. Each cell prints ✅ if everything is working or ❌ with a fix instruction. Complete every fix before moving to Week 1.

---

## ⚠️ Common Mistakes

| Mistake | What goes wrong | Fix |
|---------|-----------------|-----|
| Using 0.5 as the default threshold without justification | 0.5 is not a business decision — it is a default. The right threshold depends on what FP and FN cost. | Calculate the business value at three thresholds (0.3, 0.5, 0.7) and choose explicitly |
| Reporting only one metric in the model card | A model card without precision, recall, F1, and the chosen threshold is incomplete. | Model card must include: metric table, threshold, business rationale for threshold choice |

## 🏆 Challenge Task

> 🎯 **Core Path**
> Set your decision threshold deliberately. Show the precision-recall tradeoff curve. Mark your chosen threshold and explain the business reason in one sentence.

> 🔬 **Advanced Path**
> Implement the business value formula: value = (TP × retention_value) - (FP × call_cost). Find the threshold that maximises total value.

## ✅ What You Can Do After This Week

- Sweep decision thresholds and select the optimal one using cost-benefit logic
- Compute break-even precision and interpret it as a business constraint
- Cap offer volume by prioritising highest-probability customers
- Build a three-level sensitivity analysis (100%, 80%, 60% effectiveness)
- Produce a complete model evaluation report with confidence intervals

*Add `week8_evaluation.ipynb` to your internship portfolio.*

<a href="https://colab.research.google.com/github/internshipinabook/data-science-internshipinabook/blob/main/notebooks/week8_evaluation_STARTER.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>&nbsp;&nbsp;
<a href="https://github.com/internshipinabook/data-science-internshipinabook/blob/main/notebooks/week8/week8_evaluation_STARTER.ipynb">
  <img src="https://img.shields.io/badge/View%20on-GitHub-181717?logo=github" alt="View on GitHub"/>
</a>

---

## 🖥️ How to Run This Notebook

| Platform | Instructions |
|----------|-------------|
| **Google Colab** | Click the badge above — free, no setup needed |
| **Local Jupyter** | `pip install -r requirements.txt` then `jupyter lab` |
| **VS Code** | Open `.ipynb` with the Jupyter extension installed |
| **GitHub** | Click "View on GitHub" above for a read-only preview |

> **STARTER notebook** — Complete the `# YOUR CODE HERE` cells. Check the SOLUTION notebook if stuck.

In [ ]:
# ── PLATFORM SETUP ───────────────────────────────────────────────────────────
# Run this cell first. It installs missing libraries in Google Colab.
# In a local environment, skip it if requirements.txt is already installed.

import sys, subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("📦 Google Colab detected — installing libraries...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'pandas>=1.5', 'numpy>=1.23', 'matplotlib>=3.6',
                    'seaborn>=0.12', 'scikit-learn>=1.2'], check=True)
    print("✅ Libraries installed.")
else:
    print("💻 Running locally.")
    print("   If you see ImportError below, run: pip install -r requirements.txt")

## 📂 Dataset

The dataset is included in the course repository. No Kaggle account required.

**File:** `customer_churn.csv`  
**Source:** `https://raw.githubusercontent.com/internshipinabook/data-science-internshipinabook/main/data/customer_churn.csv`

Run the cell below to load it directly.

## 🔄 Adaptability — Use Your Own Dataset

This notebook is written for the Telco Churn dataset but the **entire ML pipeline works on any binary classification problem**.

**To swap in your own data, change only these lines at the top of the notebook:**
```python
DATA_FILE      = 'customer_churn.csv'  # ← your CSV filename
TARGET_COL     = 'Churn'              # ← the binary column you want to predict (Yes/No or 1/0)
POSITIVE_LABEL = 'Yes'                # ← which value means "positive" (the event you care about)
ID_COL         = 'customerID'         # ← identifier column to drop before modelling (or None)
NUMERIC_FIX    = 'TotalCharges'       # ← any numeric column stored as text (or None)
```

**Works with any binary classification dataset:**
- 🏦 Loan default prediction
- 🏥 Patient readmission risk
- 📧 Email spam detection
- 💳 Credit card fraud
- 🛒 Customer conversion
- 🎓 Student dropout prediction

> ⚠️ **Class imbalance warning:** If your dataset has <20% positive class, use `class_weight='balanced'` — already done in this notebook. Check your class distribution in Step 4 before proceeding.

---

## 🔄 Using a Different Dataset?

This notebook is written for `customer_churn.csv` but the ML pipeline applies to any binary classification problem.

**To adapt:**
1. Change `DATA_FILE` to your filename
2. Set `TARGET_COL` to your binary target column (the thing you want to predict)
3. Update `POSITIVE_LABEL` to the positive class name (e.g. `'Yes'`, `'1'`, `'Fraud'`)
4. The `prepare_dataset()` function encodes features — update the column references to match yours

```python
# ── Adapt these for your dataset ─────────────────────────────────────────────
DATA_FILE      = 'customer_churn.csv'   # ← change to your file
TARGET_COL     = 'Churn'               # ← your binary target column
POSITIVE_LABEL = 'Yes'                 # ← the positive class value
ID_COL         = 'customerID'          # ← identifier column to drop (or None)
```

**Works with any binary classification dataset** — fraud detection, disease prediction,
loan default, email spam, customer conversion — as long as your target has two classes.

---

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (recall_score, precision_score, f1_score,
                              classification_report, confusion_matrix,
                              roc_curve, auc, precision_recall_curve)
%matplotlib inline; sns.set_theme(style='whitegrid')
OFFER_COST=30; CLV_SAVED=240; MONTHLY_CAP=500
BREAK_EVEN=OFFER_COST/(OFFER_COST+CLV_SAVED)
print(f'✅ Libraries loaded | Break-even precision: {BREAK_EVEN:.3f}')

In [ ]:
def prepare_dataset(df):
    df=df.copy()
    df['TotalCharges']=pd.to_numeric(df['TotalCharges'],errors='coerce')
    df['TotalCharges']=df['TotalCharges'].fillna(0)  # new customers have zero charges
    if 'customerID' in df.columns: df=df.drop(columns=['customerID'])
    le=LabelEncoder(); df['Churn']=le.fit_transform(df['Churn'].astype(str))
    df['tenure_charge_ratio']=df['TotalCharges']/(df['tenure']+1)
    sc=[c for c in ['PhoneService','MultipleLines','InternetService','OnlineSecurity',
                     'OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
        if c in df.columns]
    for col in sc: df[f'{col}_binary']=df[col].apply(lambda x:1 if str(x).lower()=='yes' else 0)
    df['num_services']=df[[f'{c}_binary' for c in sc]].sum(axis=1)
    df['contract_numeric']=df['Contract'].map({'Month-to-month':2,'One year':1,'Two year':0})
    df['paperless_numeric']=(df['PaperlessBilling']=='Yes').astype(int)
    df['contract_paperless_interaction']=df['contract_numeric']*df['paperless_numeric']
    df['tenure_segment']=pd.cut(df['tenure'],bins=[0,12,36,float('inf')],labels=['New','Established','Loyal'])
    return df
print('✅ prepare_dataset() defined')

## Business Framework
*(Calculate before any code)*

**TP value:** $  
**FP cost:** $  
**FN cost:** $  (opportunity cost — explain the distinction)
**Break-even precision:** OFFER_COST/(OFFER_COST+CLV_SAVED) =

## 1. Load, prepare, train best model

In [ ]:
# YOUR CODE HERE


## 2. Threshold analysis table (7 thresholds, including FN/miss column)

In [ ]:
# YOUR CODE HERE


## 3. Precision-recall curve — best F1, default 0.5, break-even line

In [ ]:
# YOUR CODE HERE


## 4. cost_benefit_analysis() function (with cap and effectiveness params)

In [ ]:
def cost_benefit_analysis(y_true, y_proba, threshold,
                           offer_cost, clv_saved, cap=None, effectiveness=1.0):
    # YOUR CODE HERE
    # Return: dict with threshold, tp, fp, fn, tn, offers_sent, revenue, cost, net_value, precision, recall
    pass


## 5. Find optimal threshold — sweep 0.10–0.85

In [ ]:
# YOUR CODE HERE


## 6. Net value chart — shade profitable / loss-making ranges

In [ ]:
# YOUR CODE HERE


## 7. Cap behaviour — what happens when flagged > MONTHLY_CAP

In [ ]:
# YOUR CODE HERE


## 8. Extrapolation — scale test net value to full base, ±2 CV std range

In [ ]:
# YOUR CODE HERE


## 9. ROC curve and AUC

In [ ]:
# YOUR CODE HERE


## 10. Amara's Five Comments
*(Address each with code + written response)*

**Comment 1:** FN column in threshold table ✅ (already done above)
**Comment 2:** Effectiveness sensitivity — 100%, 80%, 60%
**Comment 3:** Confusion matrix with counts AND percentages — state miss rate
**Comment 4:** Reference CV confidence intervals
**Comment 5:** Limitations section — four items minimum

In [ ]:
# YOUR CODE HERE — comments 2–5


## 11. Six-Section Evaluation Report
### 1 — Model Summary
### 2 — Business Framework
### 3 — Performance at Recommended Threshold
### 4 — Business Value
*(Name the effectiveness assumption for planning — and why.)*
### 5 — Limitations
### 6 — Recommendations
*(Name specific monitoring metric and trigger threshold.)*

---
## ✅ Week 8 Complete
**Next:** `week9/*_STARTER.ipynb`

---
*github.com/InternshipInABook*